In [1]:
# FUNCTION CALLING & MCP

# Define the Tools (Business Functions)

# We'll create a set of mock functions that represent a real backend.

# Mock database
PRODUCT_DB = {
    "starter": 99,
    "professional": 299,
    "enterprise": "custom (contact sales)"
}

CUSTOMER_DB = {
    "acme_corp": {"tier": "enterprise", "discount": 0.15},
    "startup_inc": {"tier": "starter", "discount": 0.0}
}

def get_product_price(product_name: str) -> str:
    """Return the price of a product."""
    product_name = product_name.lower()
    if product_name in PRODUCT_DB:
        price = PRODUCT_DB[product_name]
        return f"The {product_name} plan costs ${price} per month."
    else:
        return "Product not found."

def get_customer_discount(company: str) -> str:
    """Return the discount for a given company."""
    company = company.lower()
    if company in CUSTOMER_DB:
        info = CUSTOMER_DB[company]
        return f"{company} is on the {info['tier']} tier with a {info['discount']*100}% discount."
    else:
        return "Company not found."

# Tool registry (this will be exposed to the model)
tools = [
    {
        "name": "get_product_price",
        "description": "Get the monthly price of a product plan.",
        "parameters": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string", "description": "Plan name (starter, professional, enterprise)"}
            },
            "required": ["product_name"]
        }
    },
    {
        "name": "get_customer_discount",
        "description": "Get the discount and tier for a customer company.",
        "parameters": {
            "type": "object",
            "properties": {
                "company": {"type": "string", "description": "Company name (acme_corp, startup_inc)"}
            },
            "required": ["company"]
        }
    }
]

In [ ]:
# Prompt the Model to Use Tools
# We'll augment our instruction prompt with the tool descriptions and ask the model to output a function call in 
# a structured JSON format when needed. We'll parse that and execute the function.

def build_prompt_with_tools(ground_truth, user_utterance, tools):
    tool_desc = json.dumps(tools, indent=2)
    prompt = f"""You are an AI sales coach. You can call external functions to retrieve real-time data. The available tools are listed below.

{tool_desc}

If you need to call a function, output a JSON block exactly like this:
```json
{{
  "function_call": {{
    "name": "<function_name>",
    "arguments": {{"arg_name": "value"}}
  }}
}}
```